In [0]:
%run ./00_config

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_trade_ledger (
  trade_id STRING, client_id STRING, asset_id STRING, trade_date DATE, side STRING,
  quantity DECIMAL(18,4), unit_price DECIMAL(18,4),
  position_after DECIMAL(18,4), avg_cost_after DECIMAL(18,6),
  realized_pnl DECIMAL(18,4), oversold BOOLEAN
);

CREATE TABLE IF NOT EXISTS gold_positions (
  client_id STRING, asset_id STRING,
  quantity DECIMAL(18,4), avg_cost DECIMAL(18,6), cost_basis DECIMAL(18,4),
  realized_pnl_total DECIMAL(18,4), trade_count BIGINT, oversell_count BIGINT,
  last_trade_date DATE, _updated_at TIMESTAMP
);

In [0]:
from decimal import Decimal
import pandas as pd

Q4, Q6 = Decimal("0.0001"), Decimal("0.000001")

LEDGER_COLS = ["trade_id", "client_id", "asset_id", "trade_date", "side", "quantity", "unit_price",
               "position_after", "avg_cost_after", "realized_pnl", "oversold"]
LEDGER_SCHEMA = ("trade_id string, client_id string, asset_id string, trade_date date, side string, "
                 "quantity decimal(18,4), unit_price decimal(18,4), position_after decimal(18,4), "
                 "avg_cost_after decimal(18,6), realized_pnl decimal(18,4), oversold boolean")

def wac_ledger(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values(["trade_date", "trade_id"])
    position, avg_cost = Decimal(0), Decimal(0)
    rows = []

    for t in pdf.itertuples(index=False):
        qty, price = Decimal(t.quantity), Decimal(t.unit_price)
        realized, oversold = Decimal(0), False

        if t.side == "BUY":
            new_position = position + qty
            avg_cost = (position * avg_cost + qty * price) / new_position if new_position != 0 else Decimal(0)
            position = new_position
        else:
            oversold = qty > position
            realized = qty * (price - avg_cost)
            position -= qty
            if position == 0:
                avg_cost = Decimal(0)

        rows.append((t.trade_id, t.client_id, t.asset_id, t.trade_date, t.side,
                     qty.quantize(Q4), price.quantize(Q4), position.quantize(Q4),
                     avg_cost.quantize(Q6), realized.quantize(Q4), oversold))

    return pd.DataFrame(rows, columns=LEDGER_COLS)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

PAIR = ["client_id", "asset_id"]
TRADE_COLS = ["trade_id", "client_id", "asset_id", "trade_date", "side", "quantity", "unit_price"]

def process(batch_df, batch_id):
    session = batch_df.sparkSession

    # 1. which (client, asset) pairs changed in this batch?
    pairs = batch_df.select(*PAIR).distinct()

    # 2. full trade history for just those pairs, recalculated
    trades = session.table("silver_trades").select(*TRADE_COLS).join(pairs, PAIR, "left_semi")
    ledger = trades.groupBy(*PAIR).applyInPandas(wac_ledger, schema=LEDGER_SCHEMA)

    # 3. upsert into the ledger
    (DeltaTable.forName(session, "gold_trade_ledger").alias("t")
        .merge(ledger.alias("s"), "t.trade_id = s.trade_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    # 4. summarize those pairs' ledgers into positions
    led = session.table("gold_trade_ledger").join(pairs, PAIR, "left_semi")
    w = Window.partitionBy(*PAIR).orderBy(F.col("trade_date").desc(), F.col("trade_id").desc())

    last = (led.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
               .select(*PAIR,
                       F.col("position_after").alias("quantity"),
                       F.col("avg_cost_after").alias("avg_cost"),
                       F.col("trade_date").alias("last_trade_date")))

    totals = led.groupBy(*PAIR).agg(
        F.sum("realized_pnl").cast("decimal(18,4)").alias("realized_pnl_total"),
        F.count("*").alias("trade_count"),
        F.sum(F.col("oversold").cast("int")).cast("bigint").alias("oversell_count"))

    positions = (last.join(totals, PAIR)
        .withColumn("cost_basis", (F.col("quantity") * F.col("avg_cost")).cast("decimal(18,4)"))
        .withColumn("_updated_at", F.current_timestamp())
        .select(*session.table("gold_positions").columns))

    # 5. upsert into positions
    (DeltaTable.forName(session, "gold_positions").alias("t")
        .merge(positions.alias("s"), "t.client_id = s.client_id AND t.asset_id = s.asset_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

query = (spark.readStream
           .option("readChangeFeed", "true")
           .table("silver_trades")
           .writeStream
           .foreachBatch(process)
           .option("checkpointLocation", f"{CHK}/gold_positions")
           .trigger(availableNow=True)
           .start())
query.awaitTermination()
print("gold_trade_ledger:", spark.table("gold_trade_ledger").count(),
      "| gold_positions:", spark.table("gold_positions").count())

In [0]:
latest_px = (spark.table("silver_prices")
    .groupBy("asset_id")
    .agg(F.max("price_date").alias("price_date"),
         F.max_by("close", "price_date").alias("last_close")))

valuation = (spark.table("gold_positions").filter("quantity > 0")
    .join(latest_px, "asset_id")
    .join(spark.table("silver_assets").select("asset_id", "ticker", "asset_class"), "asset_id")
    .join(spark.table("silver_clients").select("client_id", "tier", "risk_profile"), "client_id")
    .withColumn("market_value",   (F.col("quantity") * F.col("last_close")).cast("decimal(18,4)"))
    .withColumn("unrealized_pnl", (F.col("market_value") - F.col("cost_basis")).cast("decimal(18,4)"))
    .withColumn("unrealized_pct", F.round(F.col("unrealized_pnl") / F.col("cost_basis") * 100, 2)))

valuation.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_portfolio_valuation")

realized = (spark.table("gold_positions").groupBy("client_id")
    .agg(F.sum("realized_pnl_total").cast("decimal(18,4)").alias("realized_pnl")))

client_aum = (spark.table("gold_portfolio_valuation")
    .groupBy("client_id", "tier", "risk_profile")
    .agg(F.sum("market_value").cast("decimal(18,4)").alias("aum"),
         F.sum("cost_basis").cast("decimal(18,4)").alias("cost_basis"),
         F.sum("unrealized_pnl").cast("decimal(18,4)").alias("unrealized_pnl"),
         F.count("*").alias("open_positions"))
    .join(realized, "client_id", "left"))

client_aum.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_client_aum")
print("valuation rows:", valuation.count(), "| clients:", client_aum.count())

In [0]:
display(spark.table('gold_trade_ledger'))

In [0]:
display(spark.table('gold_positions').orderBy('client_id'))

In [0]:
display(spark.table('gold_portfolio_valuation').orderBy('client_id'))